In [ ]:
# ==========================================
# 0. IMPORT E IMPOSTAZIONI
# ==========================================

import json
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras import layers
from pathlib import Path

CARTELLA = Path(__file__).parent / "chefbot_data"
MODEL_PATH = CARTELLA / "chefbot_mobilenetv2.keras"

# Creazione automatica della cartella dati se non esiste
CARTELLA.mkdir(parents=True, exist_ok=True)
 
# 10 classi scelte tra le 101 di Food101 (stessi nomi del dataset e chiavi del JSON)
CLASSI = ["baklava", "churros", "pizza", "sushi", "caesar_salad",
          "hamburger", "spaghetti_carbonara", "tiramisu", "greek_salad", "pancakes"]

IMG_SIZE = 160

In [ ]:
# ==========================================
# 1. CARICAMENTO DI Food101
# ==========================================

(ds_train, ds_val), ds_info = tfds.load(
    "food101",                              # 101.000 immagini
    split=["train", "validation"],          # restituisce due dataset già divisi (75.750 e 25.250 immagini)
    shuffle_files=True,                     # mescola l'ordine dei file di origine a ogni lettura, per un training meno ordinato
    as_supervised=True,                     # restituisce una tupla (image, label) invece che un dizionario (formato richiesto da model.fit())
    with_info=True,                         # restituisce ds_info
    data_dir=str(CARTELLA / "tfds_data"),   # cartella di salvataggio del download
)
nomi_food101 = ds_info.features["label"].names  # le 101 classi: posizione nella lista = numero della label
 
# Numeri (0-100) che Food101 usa per le nostre 10 classi
ids = tf.constant([nomi_food101.index(c) for c in CLASSI], dtype=tf.int64)
 
 
def tieni_solo_nostre_classi(img, label):
    return tf.reduce_any(tf.equal(ids, label))  # True se la label è una delle 10
 
 
def prepara(img, label):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))    # porta tutte le foto alla stessa dimensione
    img = tf.cast(img, tf.uint8)            # uint8: occupa meno RAM nella cache rispetto a float32
    nuova_label = tf.argmax(tf.cast(tf.equal(ids, label), tf.int32))  # numero originale -> 0..9
    return img, nuova_label
 
 
def costruisci(ds):
    ds = ds.filter(tieni_solo_nostre_classi).map(prepara)   # tiene solo gli esempi delle nostre 10 classi
    return ds.cache().batch(32).prefetch(tf.data.AUTOTUNE)  # cache: dopo la prima epoca le immagini restano in RAM / batch + prefetch: mentre la CPU addestra su un batch (32 immagini), ne prepara già il successivo
 
 
train_ds = costruisci(ds_train.shuffle(1000))   # shuffle solo sul training
val_ds = costruisci(ds_val)

In [ ]:
# =============================================================
# 2. MODELLO: MobileNetV2 congelata + testa personalizzata
# =============================================================

base = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), 
    include_top=False,          # toglie il classificatore originale (1000 classi di ImageNet) e tiene solo il "feature extractor" convoluzionale che riconosce bordi, texture e forme
    weights="imagenet")          # pesi già addestrati su 1,4 milioni di immagini

base.trainable = False  # fase 1: pesi congelati, impara solo la nostra testa finale
 
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Rescaling(1 / 127.5, offset=-1)(inputs)  # pixel da 0-255 a -1/+1, come richiede MobileNetV2
x = base(x, training=False)                         # training=False: i layer BatchNorm restano in modalità inferenza (usano le statistiche di ImageNet)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)                          # pegne a caso il 30% dei neuroni in training, contro l'overfitting
outputs = layers.Dense(len(CLASSI), activation="softmax")(x)  # Testa personalizzata: 10 neuroni (uno per classe) con softmax = probabilità che sommano a 1

model = keras.Model(inputs, outputs)

In [ ]:
# ====================================================================
# 3. FASE 1 - Feature extraction (addestramento della parte finale)
# ====================================================================

model.compile(optimizer=keras.optimizers.Adam(1e-3),    # lr=1e-3: valore standard per addestrare solo la testa
              loss="sparse_categorical_crossentropy",   # label interi (0-9)
              metrics=["accuracy"])

model.summary()     # parametri "non-trainable" appartengono alla base congelata
 
callbacks = [
    # salva su disco solo il modello con la migliore accuracy in validazione
    keras.callbacks.ModelCheckpoint(str(MODEL_PATH), monitor="val_accuracy", save_best_only=True),
    # ferma l'addestramento se per 3 epoche la val_accuracy non migliora, e ripristina i pesi migliori
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
]

model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks)

In [ ]:
# ===========================================================================================
# 4. FASE 2 - Fine-tuning (sblocco gli ultimi layer della base, learning rate molto basso)
# ===========================================================================================

base.trainable = True
for layer in base.layers[:100]:     # sblocco dei layer dal 100 in poi
    layer.trainable = False         # i primi 100 layer (bordi, texture) restano congelati
 
# Dopo aver cambiato trainable si deve ricompilare, altrimenti la modifica non ha effetto
model.compile(optimizer=keras.optimizers.Adam(1e-5),    # lr molto basso per non distruggere i pesi pre-addestrati
              loss="sparse_categorical_crossentropy", 
              metrics=["accuracy"])

model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)
 
loss, acc = model.evaluate(val_ds)
print(f"Accuracy sul validation set: {acc:.1%}")
model.save(str(CARTELLA / "chefbot.keras"))

In [ ]:
# ========================================
# 5. Conoscenza piatti + lookup
# ========================================

JSON_PATH = CARTELLA / "conoscenza_piatti.json"

with open(str(JSON_PATH), encoding="utf-8") as f:
    # KB (knowledge base) = dizionario Python {etichetta: {nome, ingredienti, calorie, ..., descrizione}}
    KB = json.load(f)
 
def lookup(etichetta):
    """Data la predizione del modello, restituisce i metadati del piatto."""
    return KB[etichetta]
 
 
def stampa_scheda(etichetta):
    """Stampa la scheda informativa"""
    d = lookup(etichetta)
    print(f"\n=== {d['nome']} ===")
    print(d["descrizione"])
    print("Ingredienti:", ", ".join(d["ingredienti"]))
    print(f"Per 100 g: {d['calorie_kcal_100g']} kcal | proteine {d['proteine_g_100g']} g | "
          f"carboidrati {d['carboidrati_g_100g']} g | grassi {d['grassi_g_100g']} g")

In [ ]:
# ========================================
# 6. Da foto a scheda informativa
# ========================================

def chefbot_da_foto(percorso):
    p = Path(percorso)
    if not p.exists():
        print(f"\n[AVVISO] Immagine non trovata in: {p.absolute()}")
        print("Sostituisci il percorso con una foto reale per testare questa funzione.")
        return

    img = keras.utils.load_img(str(p), target_size=(IMG_SIZE, IMG_SIZE))        # carica e ridimensiona
    img_tensor = np.expand_dims(keras.utils.img_to_array(img), axis=0)          # (160,160,3) -> (1,160,160,3)
    probs = model.predict(img_tensor, verbose=0)[0]                             # [0] = prendo la prima (e unica) riga del batch
    etichetta = CLASSI[int(np.argmax(probs))]                                   # classe con la probabilità più alta
    print(f"Piatto riconosciuto: {etichetta} ({probs.max():.1%})")
    stampa_scheda(etichetta)

# Esempio:
# chefbot_da_foto(CARTELLA / "test_piatto.jpg")

In [ ]:
# =============================================
# 7. Ricerca per Umore (Transformers + FAISS)
# =============================================

import faiss
import torch
from transformers import AutoTokenizer, AutoModel
 
 
class SimpleEmbedder:
    # Modello multilingue, perché le frasi sono in italiano
    def __init__(self, nome="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.tokenizer = AutoTokenizer.from_pretrained(nome)
        self.model = AutoModel.from_pretrained(nome).eval()     # # .eval(): disattiva il dropout
 
    @torch.no_grad()        # niente gradienti: solo inferenza, si risparmia memoria
    def encode(self, testi):
        batch = self.tokenizer(testi, 
                               padding=True,    # allunga le frasi corte per formare un batch
                               truncation=True, # taglia quelle troppo lunghe
                               return_tensors="pt")
        out = self.model(**batch).last_hidden_state             # un vettore per ogni token
        mask = batch["attention_mask"].unsqueeze(-1).float()    # la maschera esclude il padding dalla media
        emb = (out * mask).sum(1) / mask.sum(1)                 # mean pooling: un vettore per frase
        return emb.numpy().astype("float32")                    # # FAISS richiede float32
 
 
embedder = SimpleEmbedder()
 
# Un testo per piatto: nome + descrizione + ingredienti
testi = [f"{KB[c]['nome']}. {KB[c]['descrizione']} Ingredienti: {', '.join(KB[c]['ingredienti'])}"
         for c in CLASSI]
emb_piatti = embedder.encode(testi)

# Dopo la normalizzazione L2 ogni vettore ha lunghezza 1, quindi il prodotto scalare (cio che calcola IndexFlatIP) è uguale alla similarità del coseno
faiss.normalize_L2(emb_piatti)  
indice = faiss.IndexFlatIP(emb_piatti.shape[1])
indice.add(emb_piatti)  # la riga i dell'indice corrisponde a CLASSI[i]
 
 
def cerca_per_umore(frase, k=3):
    """Restituisce i k piatti più vicini alla frase, con la loro similarità del coseno."""
    q = embedder.encode([frase])                # la query passa nello stesso modello dei testi
    faiss.normalize_L2(q)                       # normalizzata allo stesso modo
    punteggi, posizioni = indice.search(q, k)   # restituisce i punteggi e le posizioni dei k vicini
    print(f'\nRichiesta: "{frase}"')
    for i, s in zip(posizioni[0], punteggi[0]):
        print(f"- {KB[CLASSI[i]]['nome']} (similarità {s:.2f})")
    stampa_scheda(CLASSI[posizioni[0][0]])  # scheda del più vicino
 

# Test della ricerca per umore (2 frasi di prova)
for frase in ["dolce con miele", "voglio qualcosa di fresco e leggero"]:
    cerca_per_umore(frase)